# Conditional Edge

## 조건부 Edge(`Conditional Edge`) 사용해 분기 만들기
* `Conditional Edge`를 이용해 상황에 따라 다른 노드로 연결되도록 구현할 수 있습니다.

In [1]:
from typing import Literal, Annotated

from pydantic import BaseModel, Field

from langgraph.graph.message import add_messages

from langchain_core.messages import BaseMessage, HumanMessage

class AgentState(BaseModel):
    # Annotated[타입, 메타데이터] -> 해당 변수에 '메타데이터'에 해당하는 특성(기능)을 부여함
    # add_messages -> message에 적용하는 메타데이터, Node에서 history에 추가할 항목만 반환해도 알아서 리스트에 추가해 줌
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list) # 맨 처음 생성 시 빈 리스트로 시작
    last_player: Literal['A', 'B']
    number: int
    end: bool

In [7]:
def node_A(state: AgentState) -> dict:
    num = state.number
    new_num = num ** 2
    return {
        'chat_history': [HumanMessage(name='A', content=f"숫자를 {num}에서 {new_num}(으)로 바꿈")],
        'number': new_num,
        'last_player': 'A'
    }

In [8]:
def node_B(state: AgentState) -> dict:
    num = state.number
    new_num = num - 1
    return {
        'chat_history': [HumanMessage(name='B', content=f"숫자를 {num}에서 {new_num}(으)로 변경")],
        'number': new_num,
        'last_player': 'B'
    }

In [9]:
def node_judge(state: AgentState) -> dict:
    end = True if state.number > 100 else False
    return {
        'chat_history': [HumanMessage(name='judge', content=f"최종 숫자 {state.number}(으)로 종료합니다" if end else "계속 진행하세요")],
        'end': end
    }

In [10]:
from langgraph.graph import START, END, StateGraph

workflow = StateGraph(AgentState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('judge', node_judge)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'judge')
workflow.add_edge('B', 'judge')

## `route` 만들고 `conditional edge`에 부여하기
* `route`는 조건에 따라서 달라지는 목적지 `node`명 을 반환하는 함수입니다
* `workflow.add_conditional_edges`함수에 `route`를 적용하면, route가 반환한 `node`명으로 라우팅됩니다.

In [6]:
def judge_route(state: AgentState):
    # end가 True면 END로,
    # 짝수면 A로, 홀수면 B로 가는 route
    if state.end:
        return END
    elif state.last_player == 'B':
        return 'A'
    else:
        return 'B'

workflow.add_conditional_edges(
    'judge', 
    judge_route
)

In [7]:
app = workflow.compile()

init_input = AgentState(chat_history=[], last_player='A', number=2, end=False)

for chunk in app.stream(init_input):
    print(chunk)

{'A': {'chat_history': [HumanMessage(content='숫자를 2에서 4(으)로 바꿈', additional_kwargs={}, response_metadata={}, name='A', id='4d42ec95-3fd2-416b-b37c-9f75300e5a29')], 'number': 4, 'last_player': 'A'}}
{'judge': {'chat_history': [HumanMessage(content='계속 진행하세요', additional_kwargs={}, response_metadata={}, name='judge', id='bff16a73-cd7e-4793-a8a9-89d2a60442f5')], 'end': False}}
{'B': {'chat_history': [HumanMessage(content='숫자를 4에서 3(으)로 변경', additional_kwargs={}, response_metadata={}, name='B', id='ecc8589a-3d99-4018-b4c4-1420564de6ec')], 'number': 3, 'last_player': 'B'}}
{'judge': {'chat_history': [HumanMessage(content='계속 진행하세요', additional_kwargs={}, response_metadata={}, name='judge', id='9fe23eb5-42cb-4131-bba4-c103455760ca')], 'end': False}}
{'A': {'chat_history': [HumanMessage(content='숫자를 3에서 9(으)로 바꿈', additional_kwargs={}, response_metadata={}, name='A', id='09346ec7-4051-4bd7-a188-bd6cc2da6655')], 'number': 9, 'last_player': 'A'}}
{'judge': {'chat_history': [HumanMessage(content

## 토마토 게임 만들기
* ai와 토마토 게임을 진행할 수 있는 LangGraph 기반 게임을 만들어 봅시다
* 규칙: 두 명이 번갈아가면서 '토마토'를 한글자씩 말합니다. 올바르게 말하지 못하면 패배합니다.

In [8]:
from pydantic import BaseModel, Field
from typing import Literal, Optional

class TomatoGameState(BaseModel):
    next_player: Literal['user', 'ai']
    ...

In [ ]:
def user_node(state: TomatoGameState) -> dict:
    user_input = input("당신의 대답: ")
    return {
        'player_input': user_input
    }

def ai_node(state: TomatoGameState) -> dict:
    ...

def judge_node(state: TomatoGameState) -> dict:
    ...

In [ ]:
from typing import Annotated, Literal, Optional
from pydantic import BaseModel, Field
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage

SYLLABLES = ['토', '마', '토']  # 순서 고정

class TomatoGameState(BaseModel):
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    turn_index: int = 0                              # 다음에 말해야 할 글자 인덱스
    next_player: Literal['user', 'ai'] = 'user'      # judge 이후 갈 노드
    player_input: str = ''                           # 방금 말한 글자
    last_player: Literal['user', 'ai', ''] = ''      # 방금 말한 사람
    game_over: bool = False
    loser: Optional[Literal['user', 'ai']] = None

In [23]:
def user_node(state: TomatoGameState) -> dict:
    expected = SYLLABLES[state.turn_index % 3]
    user_input = input(f"당신 차례: ").strip()

    return {
        'player_input': user_input,
        'last_player': 'user',
        'chat_history': [HumanMessage(name='user', content=user_input)],
    }

In [24]:
def ai_node(state: TomatoGameState) -> dict:
    answer = SYLLABLES[state.turn_index % 3]  # 정답을 그대로 말함

    return {
        'player_input': answer,
        'last_player': 'ai',
        'chat_history': [HumanMessage(name='ai', content=answer)],
    }

In [25]:
def judge_node(state: TomatoGameState) -> dict:
    expected = SYLLABLES[state.turn_index % 3]
    actual = state.player_input.strip()

    # 틀린 경우 → 게임 종료
    if actual != expected:
        loser = state.last_player
        return {
            'game_over': True,
            'loser': loser,
            'next_player': state.next_player,  # route에서 END로 보냄
            'chat_history': [
                HumanMessage(
                    name='judge',
                    content=f"❌ 틀렸습니다! '{actual}' (정답: '{expected}') → {loser} 패배!"
                )
            ],
        }

    # 맞은 경우 → turn_index +1, 다음 플레이어 지정
    new_turn = state.turn_index + 1
    next_player = 'ai' if state.last_player == 'user' else 'user'
    next_expected = SYLLABLES[new_turn % 3]

    return {
        'turn_index': new_turn,
        'next_player': next_player,
        'game_over': False,
        'chat_history': [
            HumanMessage(
                name='judge',
                content=f"✅ '{actual}' 정답! 다음: {next_player} → '{next_expected}'"
            )
        ],
    }

In [26]:
from langgraph.graph import START, END, StateGraph

def judge_route(state: TomatoGameState):
    if state.game_over:
        return END
    return state.next_player   # 'user' 또는 'ai'

In [27]:
tomato_workflow = StateGraph(TomatoGameState)

tomato_workflow.add_node('user', user_node)
tomato_workflow.add_node('ai', ai_node)
tomato_workflow.add_node('judge', judge_node)

tomato_workflow.add_edge(START, 'user')
tomato_workflow.add_edge('user', 'judge')
tomato_workflow.add_edge('ai', 'judge')
tomato_workflow.add_conditional_edges('judge', judge_route)

tomato_app = tomato_workflow.compile()

In [28]:
init_state = TomatoGameState()

for chunk in tomato_app.stream(init_state):
    print(chunk)

{'user': {'player_input': '토', 'last_player': 'user', 'chat_history': [HumanMessage(content='토', additional_kwargs={}, response_metadata={}, name='user', id='1625b376-60f7-41f9-b732-2d20333fed38')]}}
{'judge': {'turn_index': 1, 'next_player': 'ai', 'game_over': False, 'chat_history': [HumanMessage(content="✅ '토' 정답! 다음: ai → '마'", additional_kwargs={}, response_metadata={}, name='judge', id='8b3d9d14-af2d-4787-988f-2ede77f015f3')]}}
{'ai': {'player_input': '마', 'last_player': 'ai', 'chat_history': [HumanMessage(content='마', additional_kwargs={}, response_metadata={}, name='ai', id='f5c55fdf-b3fe-4c20-a283-342982e694e0')]}}
{'judge': {'turn_index': 2, 'next_player': 'user', 'game_over': False, 'chat_history': [HumanMessage(content="✅ '마' 정답! 다음: user → '토'", additional_kwargs={}, response_metadata={}, name='judge', id='bb2f7e13-666c-47fe-ae20-00abac6d9a54')]}}
{'user': {'player_input': '토', 'last_player': 'user', 'chat_history': [HumanMessage(content='토', additional_kwargs={}, response_

In [29]:
from pydantic import BaseModel
from typing import Optional, Any

class FanOutState(BaseModel):
    value1: Optional[Any] = None
    value2: Optional[Any] = None

In [30]:
import time

def node_A(state: FanOutState) -> dict:
    print("A 노드에서 출발합니다.")
    return {"from_node": "A"}

def node_B(state: FanOutState) -> dict:
    for i in range(1, 4):
        print(f"B 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value1": state.value1 + 1}

def node_C(state: FanOutState) -> dict:
    for i in range(1, 4):
        print(f"C 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value2": state.value2 + 1}

In [31]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(FanOutState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')

app = workflow.compile()

In [32]:
init_state = FanOutState(value1=0, value2=0)
result = app.invoke(init_state)
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/3)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/3)C 노드 실행 중... (2/3)

B 노드 실행 중... (3/3)
C 노드 실행 중... (3/3)


{'value1': 1, 'value2': 1}

## 팬인(Fan-In) 구현하기
* `add_edge` 함수의 출발점은 여러 노드의 배열도 입력받을 수 있습니다.

In [33]:
def node_D(state: FanOutState):
    for i in range(1, 4):
        print(f"D 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value2": state.value1 + 1}

In [35]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(FanOutState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')
workflow.add_edge(['B', 'C'], 'D')
workflow.add_edge('D', END)

app = workflow.compile()

In [36]:
init_state = FanOutState(value1=0, value2=0)
result = app.invoke(init_state)
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/3)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/3)C 노드 실행 중... (2/3)

B 노드 실행 중... (3/3)C 노드 실행 중... (3/3)

D 노드 실행 중... (1/3)
D 노드 실행 중... (2/3)
D 노드 실행 중... (3/3)


{'value1': 1, 'value2': 2}

### Q. 팬인 시 노드 종료 시점이 다르다면?

In [37]:
import time

def node_A(state: FanOutState) -> dict:
    print("A 노드에서 출발합니다.")
    return {"from_node": "A"}

def node_B(state: FanOutState) -> dict:
    for i in range(1, 4):
        print(f"B 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value1": state.value1 + 1}

def node_C(state: FanOutState) -> dict:
    for i in range(1, 4):
        print(f"C 노드 실행 중... ({i}/3)")
        time.sleep(2)
    return {"value2": state.value2 + 1}

In [39]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(FanOutState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')
workflow.add_edge(['B', 'C'], 'D')
workflow.add_edge('D', END)

app = workflow.compile()

In [40]:
init_state = FanOutState(value1=0, value2=0)
result = app.invoke(init_state)
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/3)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/3)
C 노드 실행 중... (2/3)B 노드 실행 중... (3/3)

C 노드 실행 중... (3/3)
D 노드 실행 중... (1/3)
D 노드 실행 중... (2/3)
D 노드 실행 중... (3/3)


{'value1': 1, 'value2': 2}

# `Reducer` 활용하기

In [41]:
import time

def node_A(state: FanOutState) -> dict:
    print("A 노드에서 출발합니다.")
    return {"from_node": "A"}

def node_B(state: FanOutState) -> dict:
    for i in range(1, 6):
        print(f"B 노드 실행 중... ({i}/5)")
        time.sleep(1)
    return {"value1": state.value1 + 1}

def node_C(state: FanOutState) -> dict: # B, C 노드 모두에서 value1 값을 변경하는 경우
    for i in range(1, 4):
        print(f"C 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value1": state.value1 + 1}

    

In [42]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(FanOutState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')
workflow.add_edge(['B', 'C'], 'D')
workflow.add_edge('D', END)

app = workflow.compile()

In [43]:
init_state = FanOutState(value1=0, value2=0)
result = app.invoke(init_state) # reducer가 없으면 오류 발생
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/5)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/5)C 노드 실행 중... (2/3)

C 노드 실행 중... (3/3)B 노드 실행 중... (3/5)

B 노드 실행 중... (4/5)
B 노드 실행 중... (5/5)


InvalidUpdateError: At key 'value1': Can receive only one value per step. Use an Annotated key to handle multiple values.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_CONCURRENT_GRAPH_UPDATE

In [44]:
from typing import Annotated

def numberAddReducer(left: int, right: int) -> int: # left: 기존 State, right: return 받은 state
    return left + right # return 받은 값으로 대체하지 않고, 기존 값에 return 받은 값을 더함

class ReducerState(BaseModel):
    value1: Annotated[Optional[Any], numberAddReducer] = None
    value2: Optional[Any] = None

In [45]:
def node_B(state: ReducerState) -> dict: # state 정의도 모두 변경
    for i in range(1, 6):
        print(f"B 노드 실행 중... ({i}/5)")
        time.sleep(1)
    return {"value1": 1} # 새로 더해줄 값만 반환하면 됨

def node_C(state: ReducerState) -> dict: # B, C 노드 모두에서 value1 값을 변경하는 경우
    for i in range(1, 4):
        print(f"C 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value1": 1}

def node_D(state: ReducerState):
    for i in range(1, 4):
        print(f"D 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value2": 1}


In [46]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ReducerState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')
workflow.add_edge(['B', 'C'], 'D')
workflow.add_edge('D', END)

app = workflow.compile()

In [47]:
init_state = ReducerState(value1=0, value2=0)
result = app.invoke(init_state) # reducer가 없으면 오류 발생
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/5)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/5)
C 노드 실행 중... (2/3)
B 노드 실행 중... (3/5)
C 노드 실행 중... (3/3)
B 노드 실행 중... (4/5)
B 노드 실행 중... (5/5)
D 노드 실행 중... (1/3)
D 노드 실행 중... (2/3)
D 노드 실행 중... (3/3)


{'value1': 2, 'value2': 1}

### 동적 팬아웃 구현하기
* `Conditional Edge`를 이용하면 Fan-Out을 동적으로 설정할 수 있습니다
* Fan-Out을 설정하려면, 목적지 노드 이름을 리스트로 모두 전달하면 됩니다.

In [48]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ReducerState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')

def fanout_router(state: ReducerState):
    if state.value2 > 0:
        return 'B'
    else:
        return ['B', 'C']
    
workflow.add_conditional_edges('A', fanout_router)

workflow.add_edge(['B', 'C'], 'D')

def end_router(state: ReducerState):
    if state.value1 > 2:
        return END
    else:
        return 'A'

workflow.add_conditional_edges('D', end_router)


app = workflow.compile()

In [49]:
init_state = ReducerState(value1=0, value2=0)
result = app.invoke(init_state)
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/5)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/5)C 노드 실행 중... (2/3)

C 노드 실행 중... (3/3)B 노드 실행 중... (3/5)

B 노드 실행 중... (4/5)
B 노드 실행 중... (5/5)
D 노드 실행 중... (1/3)
D 노드 실행 중... (2/3)
D 노드 실행 중... (3/3)
A 노드에서 출발합니다.
B 노드 실행 중... (1/5)
B 노드 실행 중... (2/5)
B 노드 실행 중... (3/5)
B 노드 실행 중... (4/5)
B 노드 실행 중... (5/5)


{'value1': 3, 'value2': 1}

### 라우팅 심화 문제
* Fan-Out과 Reducing, 조건부 Edge를 적극적으로 활용해 음식점 운영을 최적화해봅시다 (괄호 안의 숫자는 소요시간(초))
```
burger: patty(5), bun(3) 각각 필요 -> burgerSet(2)
fries: fry(3) -> friesSet(1)
beverage: beverage(2)

세 가지가 모두 준비되면 종료
```

In [2]:
from pydantic import BaseModel
from typing import List, Annotated, Literal, Optional

# def menuReducer(left:List[str], right:List[str]) -> List[str]:
#     return left + right

# class RestaurantState(BaseModel):
#     menu_ordered: List[Literal['burger', 'fries', 'beverage']]
#     menu_complete: Annotated[Optional[List[Literal['burger', 'fries', 'beverage']]], menuReducer] = []

#State (이미 있음, reducer만 보강 권장)

def menuReducer(left: List[str] | None, right: List[str] | None) -> List[str]:
    return (left or []) + (right or [])

class RestaurantState(BaseModel):
    menu_ordered: List[Literal['burger', 'fries', 'beverage']]
    menu_complete: Annotated[List[str], menuReducer] = Field(default_factory=list)


In [3]:
import time

# 각종 그래프 노드 실행 및 상태 갱신 통합: 반환값으로 state의 menu_complete를 적절히 업데이트하도록 구성
def g_burger_start(state: RestaurantState) -> dict:
    print("버거 준비 시작")
    return {}

def g_fries_start(state: RestaurantState) -> dict:
    print("감자튀김 준비 시작")
    return {}

def g_patty(state: RestaurantState) -> dict:
    print("패티 조리 시작")
    time.sleep(5)
    print("패티 조리 완료")
    return {}

def g_bun(state: RestaurantState) -> dict:
    print("번 조리 시작")
    time.sleep(3)
    print("번 조리 완료")
    return {}

def g_burger_set(state: RestaurantState) -> dict:
    print("버거 세트 조립 시작")
    time.sleep(2)
    print("버거 세트 조립 완료")
    return {}

def g_burger_complete(state: RestaurantState) -> dict:
    print("버거 완성")
    return {'menu_complete': ['burger']}

def g_fry(state: RestaurantState) -> dict:
    print("감자튀김 조리 시작")
    time.sleep(3)
    print("감자튀김 조리 완료")
    return {}

def g_fries_set(state: RestaurantState) -> dict:
    print("감자튀김 세트 준비 시작")
    time.sleep(1)
    print("감자튀김 세트 준비 완료")
    return {}

def g_fries_complete(state: RestaurantState) -> dict:
    print("감자튀김 완성")
    return {'menu_complete': ['fries']}

def g_beverage(state: RestaurantState) -> dict:
    print("음료 준비 시작")
    time.sleep(2)
    print("음료 준비 완료")
    return {'menu_complete': ['beverage']}

2. 주문 확인 → 동적 Fan-Out (conditional_edges)

In [4]:
def order_router(state: RestaurantState):
    routes = []
    if 'burger' in state.menu_ordered:
        routes.append('burger_start')
    if 'fries' in state.menu_ordered:
        routes.append('fries_start')
    if 'beverage' in state.menu_ordered:
        routes.append('beverage')
    return routes  # 예: ['burger_start', 'fries_start', 'beverage']

3. 버거: Fan-Out → Fan-In → 완료

In [14]:
def g_burger_complete(state: RestaurantState) -> dict:
    print("버거 완성")
    return {'menu_complete': ['burger']}

4. 감자튀김 / 음료 (각각 독립 파이프라인)

In [15]:
def g_burger_complete(state: RestaurantState) -> dict:
    print("버거 완성")
    return {'menu_complete': ['burger']}

5. 종료 조건 (check_done)

In [16]:
def g_check_done(state: RestaurantState) -> dict:
    ordered = set(state.menu_ordered)
    completed = set(state.menu_complete)
    if ordered <= completed:
        print("🎉 모든 주문 완료!")
    return {}

def done_router(state: RestaurantState):
    if set(state.menu_ordered) <= set(state.menu_complete):
        return END
    return END  # 이 브랜치만 종료, 다른 병렬 브랜치는 계속 실행

6. 전체 그래프 조립 예시

In [17]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(RestaurantState)

# 노드 등록
for name, fn in [
    ('burger_start', g_burger_start),
    ('patty', g_patty),
    ('bun', g_bun),
    ('burger_set', g_burger_set),
    ('burger_complete', g_burger_complete),
    ('fries_start', g_fries_start),
    ('fry', g_fry),
    ('fries_set', g_fries_set),
    ('fries_complete', g_fries_complete),
    ('beverage', g_beverage),
    ('check_done', g_check_done),
]:
    workflow.add_node(name, fn)

# START → 동적 Fan-Out
workflow.add_conditional_edges(START, order_router)

# Burger 파이프라인
workflow.add_edge('burger_start', 'patty')
workflow.add_edge('burger_start', 'bun')
workflow.add_edge(['patty', 'bun'], 'burger_set')
workflow.add_edge('burger_set', 'burger_complete')
workflow.add_edge('burger_complete', 'check_done')

# Fries 파이프라인
workflow.add_edge('fries_start', 'fry')
workflow.add_edge('fry', 'fries_set')
workflow.add_edge('fries_set', 'fries_complete')
workflow.add_edge('fries_complete', 'check_done')

# Beverage
workflow.add_edge('beverage', 'check_done')

# 종료
workflow.add_conditional_edges('check_done', done_router)

app = workflow.compile()

7. 실행 예시

In [24]:
init = RestaurantState(menu_ordered=['burger', 'fries', 'beverage'])
result = app.invoke(init)
print(result)
# menu_complete: ['burger', 'fries', 'beverage'] (순서는 실행 타이밍에 따라 다를 수 있음)

음료 준비 시작
버거 준비 시작
감자튀김 준비 시작
음료 준비 완료
번 조리 시작
감자튀김 조리 시작
패티 조리 시작
번 조리 완료감자튀김 조리 완료

패티 조리 완료
버거 세트 조립 시작
감자튀김 세트 준비 시작
감자튀김 세트 준비 완료
버거 세트 조립 완료
버거 완성
감자튀김 완성
🎉 모든 주문 완료!
{'menu_ordered': ['burger', 'fries', 'beverage'], 'menu_complete': ['beverage', 'burger', 'fries']}
